# 예제 04. 예측 결과와 오류 사례 확인
빅데이터프로그래밍 · 7주차

## 목표
- 맞게 예측한 이미지와 틀리게 예측한 이미지를 각각 본다
- 숫자별 정확도를 확인한다
- 혼동행렬을 만든다

정확도 숫자 하나만 보지 말고, 무엇을 틀렸는지 봐야 합니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

test_set = datasets.MNIST("./data", train=False, download=True,
                          transform=transforms.ToTensor())
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)


## 1. 모델 준비
앞 노트북에서 저장한 가중치가 있으면 불러오고, 없으면 간단히 학습합니다.


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )
    def forward(self, x):
        return self.net(x)


model = MLP().to(device)

import os
if os.path.exists("mnist_mlp.pt"):
    model.load_state_dict(torch.load("mnist_mlp.pt", map_location=device))
    print("저장된 가중치를 불러왔습니다")
else:
    print("가중치가 없어 5 epoch 학습합니다")
    train_set = datasets.MNIST("./data", train=True, download=True,
                               transform=transforms.ToTensor())
    train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
    loss_fn, opt = nn.CrossEntropyLoss(), torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for epoch in range(5):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"  epoch {epoch+1} 완료")
    torch.save(model.state_dict(), "mnist_mlp.pt")


## 2. 전체 예측 모으기


In [ ]:
model.eval()
all_pred, all_true, all_img, all_prob = [], [], [], []

with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        prob = torch.softmax(out, dim=1)
        all_pred.append(out.argmax(dim=1).cpu())
        all_prob.append(prob.max(dim=1).values.cpu())
        all_true.append(y)
        all_img.append(x)

pred = torch.cat(all_pred); true = torch.cat(all_true)
prob = torch.cat(all_prob); imgs = torch.cat(all_img)

acc = (pred == true).float().mean().item()
print(f"시험 정확도: {acc:.4f}  ({(pred == true).sum().item()} / {len(true)})")


## 3. 맞게 예측한 이미지


In [ ]:
correct_idx = (pred == true).nonzero().flatten()[:10]

fig, axes = plt.subplots(1, 10, figsize=(16, 2.2))
for ax, i in zip(axes, correct_idx):
    ax.imshow(imgs[i].squeeze(), cmap="gray")
    ax.set_title(f"{pred[i].item()}", fontsize=11, color="green")
    ax.axis("off")
plt.suptitle("correct predictions", y=1.1)
plt.tight_layout(); plt.show()


## 4. 틀리게 예측한 이미지
사람이 봐도 애매한 글씨가 많습니다.


In [ ]:
wrong_idx = (pred != true).nonzero().flatten()
print("틀린 개수:", len(wrong_idx))

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, i in zip(axes.flatten(), wrong_idx[:16]):
    ax.imshow(imgs[i].squeeze(), cmap="gray")
    ax.set_title(f"예측 {pred[i].item()} / 정답 {true[i].item()}", fontsize=10, color="crimson")
    ax.axis("off")
plt.suptitle("wrong predictions", y=1.02)
plt.tight_layout(); plt.show()


## 5. 확신했는데 틀린 경우
확률이 높은데 틀린 것은 모델이 잘못 배운 부분입니다.


In [ ]:
conf_wrong = wrong_idx[prob[wrong_idx].argsort(descending=True)][:8]

fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for ax, i in zip(axes, conf_wrong):
    ax.imshow(imgs[i].squeeze(), cmap="gray")
    ax.set_title(f"{pred[i].item()}≠{true[i].item()}\n{prob[i].item():.2f}", fontsize=10, color="crimson")
    ax.axis("off")
plt.tight_layout(); plt.show()


## 6. 숫자별 정확도
전체 정확도가 높아도 특정 숫자만 계속 틀리는 경우가 있습니다.


In [ ]:
import pandas as pd

rows = []
for d in range(10):
    mask = true == d
    rows.append({"숫자": d, "개수": mask.sum().item(),
                 "정확도": round((pred[mask] == d).float().mean().item(), 4)})
df = pd.DataFrame(rows)
print(df.to_string(index=False))

plt.bar(df["숫자"], df["정확도"])
plt.ylim(0.9, 1.0); plt.xticks(range(10))
plt.xlabel("digit"); plt.ylabel("accuracy"); plt.title("per-class accuracy")
plt.show()


## 7. 혼동행렬 — 무엇을 무엇으로 착각하는가


In [ ]:
cm = torch.zeros(10, 10, dtype=torch.int32)
for t, p in zip(true, pred):
    cm[t, p] += 1

plt.figure(figsize=(6.5, 5.5))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xlabel("predicted"); plt.ylabel("true")
plt.xticks(range(10)); plt.yticks(range(10))
plt.title("confusion matrix")
plt.show()

# 가장 많이 헷갈린 쌍
off = cm.clone()
off.fill_diagonal_(0)
flat = off.flatten().argsort(descending=True)[:5]
print("가장 많이 헷갈린 쌍:")
for f in flat:
    t, p = divmod(f.item(), 10)
    print(f"  정답 {t} → 예측 {p} : {off[t, p].item()}회")


## 직접 해보기
1. 정답이 9인데 틀린 이미지만 골라 그려 보세요.
2. 예측 확률이 0.5 미만인 이미지는 몇 개인가요? 그 중 몇 개가 실제로 틀렸나요?


In [ ]:
# 여기에 작성하세요
